# Aula 09 - Padrão de Projeto de Metacognição


## Configuração

Este notebook demonstra o padrão de design Metacognição usando o Microsoft Agent Framework.

**Pré-requisitos:**
- Implantação do Azure OpenAI configurada via variáveis de ambiente
- Azure CLI autenticado (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## O que é Metacognição?

Metacognição é **pensar sobre o pensamento**. No contexto de agentes de IA, significa construir agentes que podem:

- **Autorrefletir** sobre suas próprias saídas e processo de raciocínio
- **Detectar erros** e se recuperar graciosamente em vez de falhar silenciosamente
- **Avaliar** se suas respostas são completas e úteis
- **Adaptar** sua estratégia quando uma abordagem inicial não funciona (ex.: recorrer a um sistema de backup)

Um agente metacognitivo não apenas responde perguntas — ele monitora seu próprio desempenho e se ajusta em tempo real.


## Ferramentas Primárias e de Backup

Um padrão comum de metacognição é a **estratégia de fallback**. O agente tenta primeiro uma ferramenta primária; se ela falhar (por exemplo, um erro 404), o agente reconhece a falha e muda de forma transparente para uma ferramenta de backup.

Isso espelha sistemas do mundo real onde serviços primários podem estar indisponíveis e agentes precisam autodiagnosticar o problema antes de escolher um caminho alternativo.

Abaixo definimos duas ferramentas de consulta de voos:
- **Primária** — cobre Paris, Tóquio e Barcelona
- **Backup** — cobre Berlim, Sydney e Nova Iorque


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Agente Auto-Reflexivo com Recuperação de Erros

O agente abaixo foi instruído a tentar primeiro o sistema de voo primário, reconhecer falhas e, de forma transparente, recorrer ao sistema de backup. Depois de cada resposta, ele avalia brevemente se respondeu completamente à pergunta do usuário.


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Padrão de Autoavaliação

Outra faceta da metacognição é **autoavaliação**: um agente separado (ou o mesmo agente em uma segunda passagem) revisa uma resposta quanto à completude, precisão e utilidade.

Abaixo criamos um agente `ResponseEvaluator` que avalia respostas de agentes de viagem em três dimensões.


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Resumo

Nesta lição você aprendeu a construir **agentes metacognitivos** usando o Microsoft Agent Framework:

- **Autorreflexão**: Agentes que monitoram seu próprio raciocínio e comunicam de forma transparente o que aconteceu.
- **Recuperação de erros com alternativas**: Um padrão de ferramenta primária + de backup em que o agente detecta falhas (por exemplo, erros 404) e automaticamente tenta uma fonte alternativa.
- **Autoavaliação**: Um agente avaliador separado que avalia as respostas quanto à completude, precisão e utilidade.

Esses padrões tornam os agentes mais robustos, transparentes e confiáveis — qualidades críticas para implantações em produção.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Isenção de responsabilidade:
Este documento foi traduzido usando o serviço de tradução por IA Co-op Translator (https://github.com/Azure/co-op-translator). Embora nos esforcemos pela precisão, esteja ciente de que traduções automatizadas podem conter erros ou imprecisões. O documento original em seu idioma nativo deve ser considerado a fonte autoritativa. Para informações críticas, recomenda-se tradução humana profissional. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas decorrentes do uso desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
